# Préparation Finale — Features & Cibles
Entrée : `features_encodees.parquet` → Sortie : `X.parquet` + `y.parquet`

- Sélection des 5 cibles `out.*`
- Transformation logarithmique (log1p) des colonnes asymétriques
- Gestion des NaN résiduels
- Séparation features / cibles

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'features_encodees.parquet')
print('Dataset chargé :', df.shape)

Dataset chargé : (549971, 422)


## 1. Sélection des cibles

In [2]:
TARGETS = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh',
    'out.emissions.total.lrmer_mid_case_25..co2e_kg',
]

TARGETS = [c for c in TARGETS if c in df.columns]

out_all  = [c for c in df.columns if c.startswith('out.')]
out_drop = [c for c in out_all if c not in TARGETS]

df.drop(columns=out_drop, inplace=True)
print(f'Cibles conservées : {len(TARGETS)}')
print('Taille Dataset  :', df.shape)


Cibles conservées : 5
Taille Dataset  : (549971, 207)


## 3. Gestion des NaN résiduels
- Colonnes > 30 % NaN → supprimées
- NaN restants (numériques) → médiane
- Lignes encore NaN → supprimées

In [3]:
nan_pct = df.isna().mean()

drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()
df.drop(columns=drop_nan_cols, inplace=True)
print(f'Colonnes supprimées (>30% NaN) : {len(drop_nan_cols)}')

nan_pct  = df.isna().mean()
num_cols = df.select_dtypes(include=[np.number]).columns
medians  = df[num_cols].median()
fill_cols = nan_pct[nan_pct > 0].index.intersection(num_cols)
df[fill_cols] = df[fill_cols].fillna(medians[fill_cols])
print(f'Colonnes imputées (médiane)    : {len(fill_cols)}')

rows_before = len(df)
df.dropna(inplace=True)
print(f'Lignes supprimées (NaN restants): {rows_before - len(df)}')
print('Taille Dataset  :', df.shape)

Colonnes supprimées (>30% NaN) : 14

Colonnes imputées (médiane)    : 5
Lignes supprimées (NaN restants): 0
Taille Dataset  : (549971, 193)


In [4]:
#df = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
print("Chargé :", df.shape)

#Colonnes string / object résiduelles 
str_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"\nColonnes string à supprimer ({len(str_cols)}) :")
print(str_cols)
df.drop(columns=str_cols, inplace=True)

#Colonnes entièrement vides (100 % NaN) 
empty_cols = df.columns[df.isna().all()].tolist()
print(f"\nColonnes vides ({len(empty_cols)}) : {empty_cols}")
df.drop(columns=empty_cols, inplace=True)

#Colonnes quasi-constantes (variance ≈ 0 ou unique value > 99.5 %) 
quasi_const = []
for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_freq >= 0.995:
        quasi_const.append(col)

print(f"\nColonnes quasi-constantes ({len(quasi_const)}) :")
print(quasi_const)
df.drop(columns=quasi_const, inplace=True)

# Colonnes dupliquées (valeurs identiques)
dup_cols = []
seen = {}
for col in df.columns:
    key = tuple(df[col].values[:500])   # empreinte rapide sur les 500 premières lignes
    if key in seen:
        dup_cols.append(col)
    else:
        seen[key] = col

print(f"\nColonnes dupliquées ({len(dup_cols)}) : {dup_cols}")
df.drop(columns=dup_cols, inplace=True)

#Résumé
print(f"\n df final : {df.shape}")
print(df.dtypes.value_counts())
print('Taille Dataset  :', df.shape)

Chargé : (549971, 193)

Colonnes string à supprimer (0) :
[]

Colonnes vides (0) : []



Colonnes quasi-constantes (20) :
['upgrade', 'weight', 'in.hvac_secondary_heating_partial_space_conditioning', 'in.lighting_interior_use', 'in.lighting_other_use', 'moisture_AK', 'in.wall_finish_fiber_cement', 'heat_fuel_Other Fuel', 'sec_heat_type_Non-Ducted Heating', 'sec_heat_type_None', 'sec_heat_fuel_Natural Gas', 'sec_heat_fuel_None', 'sec_heat_fuel_Wood', 'in.solar_collector_area', 'in.water_heater_technology_solar', 'in.water_heater_fuel_other', 'in.water_heater_fuel_solar', 'in.water_heater_fuel_wood', 'cooking_type_none', 'in.pool_heat_pump']

Colonnes dupliquées (11) : ['in.clothes_washer_usage_level', 'in.cooking_range_usage_level', 'in.dishwasher_usage_level', 'in.insulation_rim_joist', 'in.window_back', 'in.window_left', 'in.window_right', 'hvac_cooling_type_Non-Ducted Heat Pump', 'heat_fuel_None', 'hvac_shared_eff_None', 'in.misc_pool_pump_hp']

 df final : (549971, 162)
float32    43
int8       40
int64      35
float64    28
Int8       16
Name: count, dtype: int64
Tail

## 4. Séparation X / y

In [5]:

X = df[[c for c in df.columns if c.startswith('in.')]]
Y = df[TARGETS]

print(f'X : {X.shape}')
print(f'Y : {Y.shape}')



X : (549971, 108)
Y : (549971, 5)


## 5. Sauvegarde → `X.parquet` + `y.parquet`

In [6]:
X.to_parquet(DATA_PROCESSED / 'X.parquet', index=False)
Y.to_parquet(DATA_PROCESSED / 'Y.parquet', index=False)

print(f'X sauvegardé : {DATA_PROCESSED}/X.parquet  {X.shape}')
print(f'Y sauvegardé : {DATA_PROCESSED}/Y.parquet  {Y.shape}')

X sauvegardé : C:\Users\bamdyoun\stage\FlexiMax\data\processed/X.parquet  (549971, 108)
Y sauvegardé : C:\Users\bamdyoun\stage\FlexiMax\data\processed/Y.parquet  (549971, 5)
